# Lab: Linear Regression Before and After SVD

### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: October 6, 2026

---

### Goal

This lab demonstrates a practical reason for dimensionality reduction in distributed machine learning.

We will create a **very wide synthetic regression problem**, fit a Spark linear regression model directly, then compute a **truncated SVD** and fit the regression in the much smaller latent space.

We will compare:

1. **Direct regression:** `y ~ X`
2. **SVD + regression:** `X ≈ U_k Σ_k V_kᵀ`, followed by `y ~ U_k Σ_k`

(The second line says we are using the first k columns from U, the first k columns from sigma, etc)

The second model does not retain all information in `X`. It keeps only the first `k` singular directions, so there is a speed/accuracy tradeoff.


---

## 1. Basic idea

For ordinary least squares,

$\hat\beta = \arg\min_\beta \|X\beta-y\|^2.$


SVD gives


$X \approx U_k\Sigma_kV_k^T.$

$Z = U_k\Sigma_k.$

where `k` is much smaller than the original number of features.

Then `Z` has only `k` columns. 

Instead of fitting $y \sim X$, we fit $y \sim Z.$


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression # DF version
from pyspark.mllib.linalg.distributed import RowMatrix # RDD version

# one nuance is regression can use df using ML library, while *SVD uses RDDs* from MLlib library
# we need different sparse vectors for each
from pyspark.ml.linalg import Vectors, DenseVector, SparseVector # DF version
from pyspark.mllib.linalg import Vectors as MLLibVectors # RDD version

import numpy as np
import math
import time

# spark = (
#     SparkSession.builder
#     .appName("SVD-vs-LinearRegression")
#     .getOrCreate()
# )

spark = (SparkSession.builder
        .appName("SVD-vs-LinearRegression")
        .master("local[*]")
        .config("spark.driver.memory", "4g") # change max heap size so this runs
        .getOrCreate())
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))  # confirm it took

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/07 15:59:08 WARN Utils: Your hostname, Beans-Legion, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/07 15:59:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/sabine/projects/distributed_computing/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/07 15:59:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/07 15:59:08 WARN Utils: 

Spark version: 4.2.0
Default parallelism: 16


---

## 2. Set configs for the experiment


In [ ]:
N = 50_000
P = 2_000
DENSITY = 0.01 # 0.01 means we want 1% of the values to be nonzero (very sparse)
K = 30
PARTITIONS = max(8, spark.sparkContext.defaultParallelism * 2)

SEED = 314
NOISE_SD = 1.0

print(f"N observations       = {N:,}")
print(f"P original features  = {P:,}")
print(f"Nonzero density      = {DENSITY:.3%}")
print(f"SVD dimensions (k)    = {K}")
print(f"Spark partitions     = {PARTITIONS}")
print(f"Approx. nonzeros     = {N*P*DENSITY:,.0f}")


N observations       = 50,000
P original features  = 2,000
Nonzero density      = 1.000%
SVD dimensions (k)    = 30
Spark partitions     = 32
Approx. nonzeros     = 1,000,000


---

## 3. Generate a wide distributed dataset

There is **hidden low-dimensional structure**: groups of columns are correlated through a small number of latent factors. 

The response depends on a few of those factors.

A truncated SVD should be able to recover much of that structure.

We generate the data using Spark transformations so the driver never holds the entire matrix.


In [ ]:
from pyspark.sql import Row

rng_seed = SEED

# Hidden low-dimensional structure used to generate the wide feature matrix.
# Students do not need to know these latent factors; SVD's job is to discover
# approximately the same structure from X itself.
latent_dim = min(20, K)

def make_row(i):
    rng = np.random.default_rng(rng_seed + i)

    # Each observation has only a small number of nonzero feature entries.
    nnz = max(1, int(P * DENSITY))
    idx = np.sort(rng.choice(P, size=nnz, replace=False))

    # Latent factors for this observation.
    z = rng.normal(0, 1, size=latent_dim)

    # Features are generated from latent factors. Features with similar
    # column positions share latent structure, creating correlated columns.
    group = idx % latent_dim
    vals = np.array([
        z[g] + 0.15 * rng.normal()
        for g in group
    ])

    # Response depends primarily on the first few latent directions.
    beta_latent = np.array([2.0/(j+1) for j in range(min(5, latent_dim))])
    y_signal = float(np.dot(z[:len(beta_latent)], beta_latent))
    y = y_signal + rng.normal(0, NOISE_SD)

    return (
        i,
        MLLibVectors.sparse( # we need 3 things about sparse vector: length (here P, num of predictions), where are the nonzeroes, what are the values of the nonzeroes
            P,
            list(map(int, idx)),
            list(map(float, vals))
        ),
        float(y)
    )

t0 = time.perf_counter()

rdd = (
    spark.sparkContext
    .parallelize(range(N), PARTITIONS) # so we can send to workers
    .map(make_row)
    .cache() # cache rdd
)

count = rdd.count()

print(f"Generated {count:,} rows in {time.perf_counter() - t0:.2f} seconds")


Generated 50,000 rows in 2.04 seconds


### Inspect one observation

Notice that the observation is a Spark `SparseVector`.


In [ ]:
row0 = rdd.take(1)[0]
print("row id:", row0[0])
print("vector type:", type(row0[1]))
print("number of features:", row0[1].size)
print("number of nonzeros:", row0[1].numNonzeros)
print("y:", row0[2]) # doing regression so we need the y


row id: 0
vector type: <class 'pyspark.mllib.linalg.SparseVector'>
number of features: 2000
number of nonzeros: <bound method SparseVector.numNonzeros of SparseVector(2000, {258: -0.2208, 455: -0.1291, 514: -0.1439, 531: 1.8561, 576: -0.4555, 745: -0.5809, 817: -0.1318, 914: -0.3238, 1025: -0.1734, 1054: 0.0722, 1111: 1.7546, 1133: 0.2917, 1343: 1.2647, 1399: 0.3041, 1413: 0.5106, 1493: 0.1864, 1606: 1.4966, 1797: -0.1786, 1802: -0.5691, 1828: -1.3244})>
y: -4.736436019477922


Downstream we will need dataframes, so next step is to convert.

---

## 4. Put the data into the format expected by Spark ML

Spark ML's `LinearRegression` expects a DataFrame with a `label` and `features` column.


In [ ]:
df = rdd.map(lambda r: Row(label=r[2], features=r[1])).toDF().cache() # r[2] holds target (the y), r[1] holds features (the predictors)
                                                                    # we cache so that we can do ????
df.count()
df.printSchema()


root
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)



---

## 5. Baseline: fit linear regression directly

The model has **P coefficients**. With 2,000 original features, Spark is solving a regression problem with 2,000 parameters.

On a small machine, this may take a long time or exhaust available memory. This is intentional.


In [6]:
# convert back to ML vector for regression
df = (
    rdd
    .map(lambda r: Row(
        label=r[2],
        features=Vectors.dense(r[1].toArray())
    ))
    .toDF()
    .cache()
)

direct_lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    fitIntercept=True,
    maxIter=50,
    regParam=0.01
)

t0 = time.perf_counter()
direct_model = direct_lr.fit(df)
direct_time = time.perf_counter() - t0

print(f"Direct regression time: {direct_time:.2f} seconds")
print(f"Number of coefficients: {len(direct_model.coefficients)}")
print(f"RMSE: {direct_model.summary.rootMeanSquaredError:.4f}")


Direct regression time: 8.22 seconds
Number of coefficients: 2000
RMSE: 1.9283


---

## 6. Compute a truncated SVD

Now we will lower the number of dimensions to first `k` singular directions (and features).

**Note:** This computation can be expensive. However, we can do it once and reuse for different models.

`RowMatrix.computeSVD` performs the distributed linear algebra.


In [ ]:
X = RowMatrix(rdd.map(lambda r: r[1])) # partition data by rows (and since its SVD, we use the RDD format)

t0 = time.perf_counter()

svd = X.computeSVD(
    k=K, # this is a choice and we should probably do this procedure with a different k to understand tradeoffs re how well it reconstructs the data (we aren't super concerned with how slow or fast it goes since we can save and reuse result?)
    computeU=True
)

svd_time = time.perf_counter() - t0

print(f"Truncated SVD time: {svd_time:.2f} seconds")
print("Number of singular values:", len(svd.s))
print("Shape of V:", svd.V.numRows, "x", svd.V.numCols)


Traceback (most recent call last):
  File "/home/sabine/projects/distributed_computing/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 262, in manager
    code = worker(sock, authenticated)
  File "/home/sabine/projects/distributed_computing/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe
26/10/07 15:59:23 WARN RowMatrix: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.
netlib-arpack: JNI_OnLoad: dlopen(libarpack.so.2) failed: libarpack.so.2: cannot open shared object file: No such file or directory


Truncated SVD time: 42.63 seconds
Number of singular values: 30
Shape of V: 2000 x 30


26/10/07 16:00:05 WARN RowMatrix: The input data was not directly cached, which may hurt performance if its parent RDDs are also uncached.


---

## 7. Look at the singular values

The singular values tell us how much variation is associated with each SVD direction.

If the singular values fall rapidly, a small `k` may provide a useful approximation of the original matrix.


In [ ]:
singular_values = np.array(svd.s)

# print singular values
for i, value in enumerate(singular_values[:min(15, K)], start=1):
    print(f"{i:2d}: {value:,.2f}")

explained = singular_values**2 / np.sum(singular_values**2) # sum of the squared singular values is the total of the eigenvalues (is this right?)
cumulative_explained = np.cumsum(explained)

print("\nCumulative fraction represented by first k components:")
for i, value in enumerate(cumulative_explained[:min(15, K)], start=1):
    print(f"{i:2d}: {value:.4f}")


 1: 32.75
 2: 32.67
 3: 32.65
 4: 32.56
 5: 32.52
 6: 32.44
 7: 32.41
 8: 32.34
 9: 32.28
10: 32.23
11: 32.17
12: 32.12
13: 32.05
14: 32.03
15: 31.97

Cumulative fraction represented by first k components:
 1: 0.0380
 2: 0.0757
 3: 0.1134
 4: 0.1509
 5: 0.1884
 6: 0.2256
 7: 0.2627
 8: 0.2997
 9: 0.3366
10: 0.3734
11: 0.4100
12: 0.4465
13: 0.4828
14: 0.5191
15: 0.5553


*Each of them is explaining about the same amount. And the first 15 explain about 56% of the variation in the data.*

---

## 8. Construct the reduced feature matrix

`svd.U` contains the left singular vectors.

Multiplying each column of `U` by its corresponding singular value gives

$
Z = U_k\Sigma_k.
$

Every observation now has only `k` features.


In [ ]:

s_values = np.array(svd.s)

# take singular values and for each row, multiply the u by the singular value and give back as a row
def make_reduced_row(u):
    z = [float(u[i] * s_values[i]) for i in range(K)]
    return Row(features=Vectors.dense(z))

t0 = time.perf_counter()

Z_rdd = svd.U.rows.map(make_reduced_row) # here is where it actually happens

y_rdd = rdd.map(lambda r: float(r[2]))

reduced_rdd = Z_rdd.zip(y_rdd).map(
    lambda z: Row(label=z[1], features=z[0].features) # label is the y, features are the predictors
)

reduced_df = reduced_rdd.toDF().cache() # convert to dataframe and cache it
reduced_df.count()

print(f"Time: {time.perf_counter() - t0:.2f} seconds")

Time: 2.74 seconds


---

## 9. Fit the regression in SVD space

The regression now has only `k` coefficients.


In [10]:
svd_lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    fitIntercept=True,
    maxIter=50,
    regParam=0.01
)

t0 = time.perf_counter()
svd_model = svd_lr.fit(reduced_df)
svd_regression_time = time.perf_counter() - t0

print(f"SVD-space regression time: {svd_regression_time:.2f} seconds")
print(f"Number of coefficients: {len(svd_model.coefficients)}")
print(f"RMSE: {svd_model.summary.rootMeanSquaredError:.4f}")


SVD-space regression time: 0.38 seconds
Number of coefficients: 30
RMSE: 2.0139


---

## 10. Compare the results


In [11]:
print("=" * 60)
print("RESULTS")
print("=" * 60)

print(f"Original dimensions:       {P:,}")
print(f"Reduced dimensions:        {K}")
print(f"Direct regression:         {direct_time:.2f} sec")
print(f"SVD:                       {svd_time:.2f} sec")
print(f"SVD-space regression:      {svd_regression_time:.2f} sec")
print()
print(f"Direct RMSE:               {direct_model.summary.rootMeanSquaredError:.4f}")
print(f"SVD-space RMSE:            {svd_model.summary.rootMeanSquaredError:.4f}")


RESULTS
Original dimensions:       2,000
Reduced dimensions:        30
Direct regression:         8.22 sec
SVD:                       42.63 sec
SVD-space regression:      0.38 sec

Direct RMSE:               1.9283
SVD-space RMSE:            2.0139


---

## 11. Exercises

### Exercise 1 — Change `k`

Run the experiment with:

```python
K = 10
K = 100
K = 300
```

How does RMSE change?

---

### Exercise 2 — Make the matrix wider

Run the experiment with:

```python
P = 500
P = 2_000
P = 5_000
```

How does the direct regression scale compared with the SVD approach?

---

### Exercise 3 — Change the density

Try:

```python
DENSITY = 0.001
DENSITY = 0.005
DENSITY = 0.01
DENSITY = 0.05
```

What happens as the matrix becomes less sparse?



---

## 12. Takeaway

The computational pattern is:

```text
                    ORIGINAL PROBLEM
                 X: N × P, P is enormous
                          │
                          │
              ┌───────────▼───────────┐
              │      truncated SVD    │
              │    X ≈ UₖΣₖVₖᵀ        │
              └───────────┬───────────┘
                          │
                          ▼
                  Z = UₖΣₖ
                  N × k matrix
                          │
                          ▼
                  Linear Regression
                    only k features
```

**Algorithmic complexity and representation matter as much as adding machines**.

Spark can distribute a huge computation, but if the algorithm is operating in a 10,000-dimensional feature space, distribution does not make that dimensionality disappear.

SVD gives us a way to change the problem .
